In [8]:
import pandas as pd

In [9]:
filepath = "nrc_data_csv/swahili_nrc.csv"
df = pd.read_csv(filepath)
df.head(12)

,word,anger,disgust,fear,joy,sadness,surprise
0,mshangao,0,0,0,0,0,0
1,abacus,0,0,0,0,0,0
2,kuacha,0,0,1,0,1,0
3,kutelekezwa,1,0,1,0,1,0
4,kuachwa,1,0,1,0,1,1
5,poa,0,0,0,0,0,0
6,kupungua,0,0,0,0,0,0
7,aba,0,0,0,0,0,0
8,abati,0,0,0,0,0,0
9,fupisha,0,0,0,0,0,0


In [10]:
from datasets import load_dataset
import pandas as pd

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
random_state = 42

# Load German evaluation data

comb_df = pd.concat(
    map(pd.read_csv, ["brighter_swahili.csv"]),
    ignore_index=True
)

comb_df["text"] = comb_df["text"].astype(str)
comb_df["emotion"] = comb_df["emotion"].astype(str)

print("Original German rows:", len(comb_df))
print("\nOriginal German class distribution:")
print(comb_df["emotion"].value_counts())


# Balance German data
target_n = comb_df["emotion"].value_counts().min()

balanced_parts = []
for emotion, group in comb_df.groupby("emotion"):
    balanced_parts.append(
        group.sample(n=target_n, random_state=random_state)
    )

comb_df = (
    pd.concat(balanced_parts, ignore_index=True)
    .sample(frac=1, random_state=random_state)
    .reset_index(drop=True)
)

print("\nBalanced German rows:", len(comb_df))
print("\nBalanced German class distribution:")
print(comb_df["emotion"].value_counts())

Original German rows: 3965

Original German class distribution:
emotion
surprise    1103
joy          951
sadness      675
anger        623
disgust      434
fear         179
Name: count, dtype: int64

Balanced German rows: 1074

Balanced German class distribution:
emotion
joy         179
fear        179
disgust     179
surprise    179
anger       179
sadness     179
Name: count, dtype: int64


In [11]:
import spacy
import random

random.seed(42)

nlp = spacy.blank("xx")

df["word"] = df["word"].astype(str).str.lower().str.strip()

df = (
    df.groupby("word", as_index=False)[target_emotions]
    .max()
)

lexicon = df.set_index("word")[target_emotions].to_dict(orient="index")

emotion_frequency = {}

for word, scores in lexicon.items():
    active_count = sum(scores[emotion] for emotion in target_emotions)
    emotion_frequency[word] = 1 / active_count if active_count > 0 else 0

predictions = []

for text in comb_df["text"]:
    count = {emotion: 0 for emotion in target_emotions}

    doc = nlp(str(text))

    for token in doc:
        candidate = token.text.lower().strip()

        if candidate in lexicon:
            word_weight = emotion_frequency[candidate]

            for emotion in target_emotions:
                count[emotion] += lexicon[candidate][emotion] * word_weight

    maxim = max(count.values())

    if maxim == 0:
        pred = "unknown"
    else:
        best_emotions = [emo for emo, score in count.items() if score == maxim]
        pred = random.choice(best_emotions)

    predictions.append(pred)

comb_df["prediction"] = predictions

In [12]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np

target_emotions = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]

def evaluate_lexicon_predictions(df, gold_col="emotion", pred_col="prediction"):
    """
    Evaluates lexicon predictions.
    Unknown predictions are counted as wrong in the main evaluation.
    Also reports coverage and covered-only performance.
    """

    eval_df = df.copy()
    eval_df[gold_col] = eval_df[gold_col].astype(str)
    eval_df[pred_col] = eval_df[pred_col].astype(str)

    # -------------------------
    # Main evaluation: unknown = wrong
    # -------------------------
    y_true = eval_df[gold_col]
    y_pred = eval_df[pred_col]

    print("=== Main evaluation: unknown counted as wrong ===")
    print("Accuracy:", round(accuracy_score(y_true, y_pred), 4))
    print("Balanced accuracy:", round(balanced_accuracy_score(y_true, y_pred), 4))
    print("Macro F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="macro"), 4))
    print("Weighted F1:", round(f1_score(y_true, y_pred, labels=target_emotions, average="weighted"), 4))
    print()

    print(classification_report(
        y_true,
        y_pred,
        labels=target_emotions,
        digits=3,
        zero_division=0
    ))

    cm = confusion_matrix(y_true, y_pred, labels=target_emotions)
    cm_df = pd.DataFrame(cm, index=target_emotions, columns=target_emotions)

    print("Confusion matrix:")
    print(cm_df)
    print()

    # -------------------------
    # Coverage
    # -------------------------
    covered_mask = eval_df[pred_col] != "unknown"
    coverage = covered_mask.mean()

    print("=== Coverage ===")
    print("Coverage:", round(coverage, 4))
    print("Covered examples:", covered_mask.sum())
    print("Total examples:", len(eval_df))
    print()

    # -------------------------
    # Covered-only evaluation
    # -------------------------
    covered_df = eval_df[covered_mask].copy()

    if len(covered_df) > 0:
        print("=== Covered-only evaluation ===")
        print("Accuracy:", round(accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Balanced accuracy:", round(balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]), 4))
        print("Macro F1:", round(f1_score(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            average="macro",
            zero_division=0
        ), 4))
        print()

        print(classification_report(
            covered_df[gold_col],
            covered_df[pred_col],
            labels=target_emotions,
            digits=3,
            zero_division=0
        ))
    else:
        print("No covered examples.")

    results = {
        "accuracy_unknown_wrong": accuracy_score(y_true, y_pred),
        "balanced_accuracy_unknown_wrong": balanced_accuracy_score(y_true, y_pred),
        "macro_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="macro", zero_division=0),
        "weighted_f1_unknown_wrong": f1_score(y_true, y_pred, labels=target_emotions, average="weighted", zero_division=0),
        "coverage": coverage,
    }

    if len(covered_df) > 0:
        results.update({
            "accuracy_covered_only": accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "balanced_accuracy_covered_only": balanced_accuracy_score(covered_df[gold_col], covered_df[pred_col]),
            "macro_f1_covered_only": f1_score(
                covered_df[gold_col],
                covered_df[pred_col],
                labels=target_emotions,
                average="macro",
                zero_division=0
            ),
        })

    return results

In [13]:
results = evaluate_lexicon_predictions(
    comb_df,
    gold_col="emotion",
    pred_col="prediction"
)

=== Main evaluation: unknown counted as wrong ===
Accuracy: 0.1304
Balanced accuracy: 0.1304
Macro F1: 0.1663
Weighted F1: 0.1663

              precision    recall  f1-score   support

       anger      0.254     0.089     0.132       179
     disgust      0.345     0.106     0.162       179
        fear      0.271     0.145     0.189       179
         joy      0.196     0.162     0.177       179
     sadness      0.321     0.235     0.271       179
    surprise      0.125     0.045     0.066       179

   micro avg      0.251     0.130     0.172      1074
   macro avg      0.252     0.130     0.166      1074
weighted avg      0.252     0.130     0.166      1074

Confusion matrix:
          anger  disgust  fear  joy  sadness  surprise
anger        16       11    13   26       16         9
disgust      13       19    12   27       15        10
fear          7        7    26   21       21        13
joy           6        2    21   29       27         8
sadness      14        9    14   